In [1]:
import os
import tifffile 
import napari
import numpy as np
from pathlib import Path
from typing import List

In [2]:
import os
import numpy as np
import tifffile
import napari
from typing import List

class InterpolationAnnotator:
    def __init__(self):
        self.viewer = napari.Viewer()  # Single persistent viewer
        self.current_mask_path = None
        
        # Keyboard controls
        @self.viewer.bind_key('s')
        def _save_and_continue(_):
            self._save_interpolated_mask()
            self._load_next_image()
                
        @self.viewer.bind_key('q')
        def _quit(_):
            print("Annotation complete!")
            self.viewer.close()

    def annotate_images(self, image_paths: List[str], output_dir: str):
        """Process images with interpolation"""
        os.makedirs(output_dir, exist_ok=True)
        self.image_paths = [
            p for p in image_paths 
            if not os.path.exists(
                os.path.join(output_dir, f"{os.path.splitext(os.path.basename(p))[0]}_interpolated.tif")
            )
        ]
        self.output_dir = output_dir
        self._load_next_image()
        napari.run()  # Block here until viewer closes

    def _load_next_image(self):
        """Load next unprocessed image"""
        if not self.image_paths:
            self.viewer.close()
            return
            
        image_path = self.image_paths.pop(0)
        image = tifffile.imread(image_path)
        self.current_mask_path = os.path.join(
            self.output_dir,
            f"{os.path.splitext(os.path.basename(image_path))[0]}_interpolated.tif"
        )
        
        self.viewer.layers.clear()
        self.viewer.add_image(image, name='Image')
        self.viewer.add_labels(
            np.zeros(image.shape, dtype=np.uint16),
            name='Labels'
        )
        print(f"\nAnnotating: {os.path.basename(image_path)}")
        print("1. Draw on some slices in 'Labels'")
        print("2. Click 'Interpolate' to generate interpolated labels")
        print("3. Press [s] to save | [q] to quit")

    def _save_interpolated_mask(self):
        """Saves the interpolated layer if it exists"""
        try:
            for layer in self.viewer.layers:
                if layer.name == 'Labels - interpolated':  # Exact match for interpolated layer
                    tifffile.imwrite(
                        self.current_mask_path,
                        layer.data.astype(np.uint16)  # Save as 16-bit TIFF
                    )
                    print(f"Saved interpolated mask to {os.path.basename(self.current_mask_path)}")
                    return
            
            print("No interpolated labels found! Did you:")
            print("- 1. Annotate slices in 'Labels' layer")
            print("- 2. Click the 'Interpolate' button?")
        except Exception as e:
            print(f"Error saving mask: {str(e)}")


In [3]:
if __name__ == "__main__":
    input_dir = r"C:\Users\josed\Desktop\StarDIST_Training\Annotation RAW" #PUT YOUR INPUT DIR HERE"
    output_dir = r"C:\Users\josed\Desktop\StarDIST_Training\Annotation Masks"
    
    
    tiff_files = [
        os.path.join(input_dir, f) 
        for f in sorted(os.listdir(input_dir))
        if f.lower().endswith(('.tif', '.tiff'))
    ]
    
    InterpolationAnnotator().annotate_images(tiff_files, output_dir)


Annotating: POOL1_POS2_2.tif
1. Draw on some slices in 'Labels'
2. Click 'Interpolate' to generate interpolated labels
3. Press [s] to save | [q] to quit
